# UNIVERSIDAD EXTERNADO DE COLOMBIA · CIENCIA DE DATOS · 2026

## Ensambles: diversidad, blending y stacking

**Predicción de la Estancia Hospitalaria con Aprendizaje Automático — SPARCS CY2021**

Autores: Daniel Vallejo Vecil · Juliana M.

---

Este cuaderno evalúa si combinar varios modelos mejora la predicción de la DEH frente al mejor modelo
individual. Se procede en tres pasos de costo creciente, con un criterio de parada explícito en cada
uno para no invertir cómputo en una estrategia que el diagnóstico previo ya descarta.

## 1. Fundamento y criterio de decisión

Un ensamble mejora sobre sus componentes en la medida en que estos **cometan errores distintos**. Si
todos los modelos base fallan en los mismos episodios, promediarlos reduce poco o nada el error: la
combinación hereda el sesgo común en lugar de cancelarlo.

Por eso el cuaderno no empieza entrenando el apilamiento, sino midiendo la diversidad. La secuencia es:

1. **Diagnóstico de diversidad.** Se entrenan los modelos base y se calcula la correlación entre sus
   residuales. *Criterio de parada:* si todas las correlaciones superan 0,95, la ganancia esperada del
   ensamble es marginal y conviene documentarlo como hallazgo en lugar de continuar.
2. **Blending.** Combinación lineal de las predicciones con pesos no negativos. Es barato —no requiere
   reentrenar— y suele capturar la mayor parte de la ganancia disponible.
3. **Stacking.** Meta-modelo entrenado sobre predicciones fuera de pliegue. Es la variante correcta
   desde el punto de vista metodológico y también la más costosa.

### Advertencia sobre la interpretabilidad

Un modelo apilado predice a partir de las salidas de sus componentes, no de las variables originales.
Las atribuciones SHAP calculadas sobre el meta-modelo cuantificarían la contribución de cada modelo
base, no la de los predictores clínicos y administrativos, que es lo que interroga la pregunta de
investigación. Si el ensamble resulta ganador por un margen estrecho, conviene conservar el mejor
modelo individual como objeto del análisis de interpretabilidad y reportar el ensamble únicamente
como referencia de desempeño alcanzable.

## 2. Configuración y carga

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib, time

from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd().parent
GOLD_PATH = PROJECT_ROOT / "data" / "3_gold"
MODELOS_PATH = PROJECT_ROOT / "modelos"
TABLAS_PATH = PROJECT_ROOT / "tablas"
MODELOS_PATH.mkdir(parents=True, exist_ok=True)
TABLAS_PATH.mkdir(parents=True, exist_ok=True)

NOTEBOOK_NUM = "07"


def guardar_tabla(tabla, nombre):
    if isinstance(tabla, pd.Series):
        tabla = tabla.to_frame()
    archivo = TABLAS_PATH / f"{NOTEBOOK_NUM}_{nombre}.csv"
    tabla.to_csv(archivo, encoding="utf-8-sig")
    print(f"Tabla guardada: {archivo.resolve()}")


# El split viene fijado desde 03_FeatureEngineering. NO se vuelve a particionar.
X_train = pd.read_parquet(GOLD_PATH / "X_train_sparcs.parquet")
X_test = pd.read_parquet(GOLD_PATH / "X_test_sparcs.parquet")
y_train = pd.read_parquet(GOLD_PATH / "y_train_sparcs.parquet")["Length of Stay"]
y_test = pd.read_parquet(GOLD_PATH / "y_test_sparcs.parquet")["Length of Stay"]

print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")

X_train: (1672271, 189)   X_test: (418068, 189)


### 2.1 Saneamiento de los nombres de columna

Las columnas generadas por `get_dummies` conservan los caracteres de las categorías originales
—espacios, dos puntos, comas y paréntesis—. XGBoost y Random Forest los aceptan, pero LightGBM los
rechaza (`Do not support special JSON characters in feature name`).

Se sanean los nombres y se conserva el mapa inverso, necesario para que las gráficas del análisis de
interpretabilidad muestren las etiquetas legibles y no los identificadores transformados. El contador
de colisiones evita que dos categorías distintas colapsen en un mismo nombre al sustituir los
caracteres especiales, situación que produciría columnas duplicadas.

In [2]:
import re


def sanear(nombres):
    """Convierte nombres de columna a caracteres seguros, evitando colisiones."""
    limpios, vistos = [], {}
    for n in nombres:
        s = re.sub(r"[^0-9a-zA-Z_]+", "_", str(n)).strip("_")
        if not s:
            s = "col"
        if s[0].isdigit():
            s = "v_" + s
        if s in vistos:
            vistos[s] += 1
            s = f"{s}__{vistos[s]}"
        else:
            vistos[s] = 0
        limpios.append(s)
    return limpios


NOMBRES_ORIGINALES = list(X_train.columns)
nombres_saneados = sanear(NOMBRES_ORIGINALES)

# Mapa inverso: se guarda para recuperar las etiquetas legibles en las gráficas SHAP
MAPA_NOMBRES = dict(zip(nombres_saneados, NOMBRES_ORIGINALES))

X_train.columns = nombres_saneados
X_test.columns = nombres_saneados

n_cambiados = sum(1 for a, b in zip(nombres_saneados, NOMBRES_ORIGINALES) if a != b)
print(f"Nombres saneados: {n_cambiados} de {len(NOMBRES_ORIGINALES)} columnas")
print(f"Columnas duplicadas tras el saneamiento: {len(nombres_saneados) - len(set(nombres_saneados))}")
print()
print("Ejemplos de transformación:")
ejemplos = [(o, s) for s, o in MAPA_NOMBRES.items() if s != o][:5]
for o, s in ejemplos:
    print(f"  {o}  ->  {s}")

joblib.dump(MAPA_NOMBRES, MODELOS_PATH / "07_mapa_nombres_columnas.joblib")
print()
print("Mapa de nombres guardado para su uso en el análisis SHAP.")

Nombres saneados: 185 de 189 columnas
Columnas duplicadas tras el saneamiento: 0

Ejemplos de transformación:
  CCSR Diagnosis_TE  ->  CCSR_Diagnosis_TE
  CCSR Procedure_TE  ->  CCSR_Procedure_TE
  APR DRG_TE  ->  APR_DRG_TE
  Hospital Service Area_Central NY  ->  Hospital_Service_Area_Central_NY
  Hospital Service Area_Finger Lakes  ->  Hospital_Service_Area_Finger_Lakes

Mapa de nombres guardado para su uso en el análisis SHAP.


### 2.2 Submuestreo para prototipado

El stacking con validación cruzada de 5 pliegues implica 5 ajustes por modelo base sobre 1,67 millones
de filas. Se recomienda ejecutar primero el cuaderno completo con `FRACCION = 0.15` para verificar el
flujo y obtener una estimación preliminar de la ganancia; solo si esta resulta apreciable conviene
repetirlo con `FRACCION = 1.0`.

El submuestreo se aplica **únicamente al entrenamiento**. El conjunto de prueba permanece completo, de
modo que las métricas siguen siendo comparables con las de los cuadernos anteriores.

In [3]:
FRACCION = 0.15   # poner 1.0 para la ejecución definitiva
SEMILLA = 42

if FRACCION < 1.0:
    idx = X_train.sample(frac=FRACCION, random_state=SEMILLA).index
    X_tr, y_tr = X_train.loc[idx], y_train.loc[idx]
    print(f"Submuestra de entrenamiento: {X_tr.shape[0]:,} filas ({FRACCION:.0%})")
else:
    X_tr, y_tr = X_train, y_train
    print(f"Entrenamiento completo: {X_tr.shape[0]:,} filas")

print(f"Prueba (siempre completa): {X_test.shape[0]:,} filas")

Submuestra de entrenamiento: 250,841 filas (15%)
Prueba (siempre completa): 418,068 filas


## 3. Modelos base

A los tres modelos de los cuadernos anteriores se añaden LightGBM y CatBoost, si están disponibles en
el entorno. La inclusión de CatBoost no es decorativa: Jain et al. (2024) reportan este algoritmo como
el de mejor desempeño sobre SPARCS, y su mecanismo de estadísticos de objetivo ordenados aborda el
mismo problema que el suavizado del *Target Encoding* de la capa Gold, pero con una corrección por
permutaciones que evita que cada observación contribuya a la media que la codifica.

Los hiperparámetros se mantienen equivalentes entre algoritmos para que las diferencias sean
atribuibles al modelo y no a su capacidad.

In [4]:
PARAMS_XGB = dict(n_estimators=400, max_depth=8, learning_rate=0.05, subsample=0.8,
                  colsample_bytree=0.8, tree_method="hist", random_state=SEMILLA, n_jobs=-1)

modelos_base = {
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=150, max_depth=20,
                                           random_state=SEMILLA, n_jobs=-1),
    "XGBoost": XGBRegressor(objective="reg:squarederror", **PARAMS_XGB),
    "XGBoost Tweedie": XGBRegressor(objective="reg:tweedie", tweedie_variance_power=1.5,
                                    **PARAMS_XGB),
}

try:
    from lightgbm import LGBMRegressor
    modelos_base["LightGBM"] = LGBMRegressor(
        n_estimators=400, max_depth=8, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, random_state=SEMILLA, n_jobs=-1, verbose=-1)
    print("LightGBM disponible")
except ImportError:
    print("LightGBM no disponible — instalar con: pip install lightgbm")

try:
    from catboost import CatBoostRegressor
    modelos_base["CatBoost"] = CatBoostRegressor(
        iterations=400, depth=8, learning_rate=0.05, random_seed=SEMILLA, verbose=0)
    print("CatBoost disponible")
except ImportError:
    print("CatBoost no disponible — instalar con: pip install catboost")

print(f"\nModelos base a evaluar: {len(modelos_base)}")

LightGBM disponible
CatBoost disponible

Modelos base a evaluar: 6


## 4. Entrenamiento y desempeño individual

In [5]:
def metricas(y_real, y_pred):
    y_pred = np.clip(y_pred, 1, None)
    return {"MAE": mean_absolute_error(y_real, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_real, y_pred))),
            "R2": r2_score(y_real, y_pred)}


entrenados, pred_test, filas = {}, {}, []

for nombre, modelo in modelos_base.items():
    t0 = time.time()
    modelo.fit(X_tr, y_tr)
    p = np.clip(modelo.predict(X_test), 1, None)

    entrenados[nombre] = modelo
    pred_test[nombre] = p
    filas.append({"modelo": nombre, **metricas(y_test, p),
                  "segundos": round(time.time() - t0, 1)})
    print(f"{nombre:18s}  MAE={filas[-1]['MAE']:.3f}  RMSE={filas[-1]['RMSE']:.3f}  "
          f"R2={filas[-1]['R2']:.4f}  ({filas[-1]['segundos']}s)")

base_df = pd.DataFrame(filas).sort_values("R2", ascending=False).reset_index(drop=True)
display(base_df.round(4))
guardar_tabla(base_df, "DesempenoModelosBase")

MEJOR_BASE = base_df.iloc[0]["modelo"]
R2_MEJOR_BASE = base_df.iloc[0]["R2"]
print(f"\nMejor modelo individual: {MEJOR_BASE} (R2={R2_MEJOR_BASE:.4f})")

Ridge               MAE=3.272  RMSE=6.519  R2=0.3834  (1.9s)
Random Forest       MAE=3.027  RMSE=6.337  R2=0.4174  (510.6s)
XGBoost             MAE=2.987  RMSE=6.204  R2=0.4416  (25.6s)
XGBoost Tweedie     MAE=2.921  RMSE=6.182  R2=0.4456  (27.7s)
LightGBM            MAE=3.014  RMSE=6.202  R2=0.4419  (10.0s)
CatBoost            MAE=3.033  RMSE=6.242  R2=0.4347  (23.5s)


,modelo,MAE,RMSE,R2,segundos
0,XGBoost Tweedie,2.9208,6.1817,0.4456,27.7
1,LightGBM,3.0139,6.2021,0.4419,10.0
2,XGBoost,2.9869,6.2036,0.4416,25.6
3,CatBoost,3.0328,6.2421,0.4347,23.5
4,Random Forest,3.0266,6.3367,0.4174,510.6
5,Ridge,3.2718,6.5190,0.3834,1.9


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\07_DesempenoModelosBase.csv

Mejor modelo individual: XGBoost Tweedie (R2=0.4456)


## 5. Diagnóstico de diversidad

Este es el paso decisivo. Se calcula la correlación entre los residuales de cada par de modelos sobre
el conjunto de prueba. Correlaciones cercanas a 1 indican que los modelos se equivocan en los mismos
episodios y en la misma dirección, con lo cual combinarlos no aporta información nueva.

In [6]:
residuales = pd.DataFrame({n: (y_test.to_numpy() - p) for n, p in pred_test.items()})
corr_res = residuales.corr()

display(corr_res.round(3))
guardar_tabla(corr_res, "CorrelacionResiduales")

sup = corr_res.where(np.triu(np.ones(corr_res.shape), k=1).astype(bool)).stack()
print(f"\nCorrelación media entre residuales: {sup.mean():.3f}")
print(f"Correlación mínima: {sup.min():.3f}  (par: {sup.idxmin()})")
print(f"Correlación máxima: {sup.max():.3f}  (par: {sup.idxmax()})")

if sup.min() > 0.95:
    print("\n>>> DIAGNÓSTICO: los modelos base son redundantes entre sí.")
    print(">>> La ganancia esperada del ensamble es marginal. Documentar como hallazgo.")
elif sup.min() > 0.85:
    print("\n>>> DIAGNÓSTICO: diversidad limitada. Ganancia esperada pequeña pero no nula.")
else:
    print("\n>>> DIAGNÓSTICO: hay diversidad aprovechable. El ensamble puede aportar.")

,Ridge,Random Forest,XGBoost,XGBoost Tweedie,LightGBM,CatBoost
Ridge,1.000,0.916,0.938,0.947,0.960,0.965
Random Forest,0.916,1.000,0.964,0.959,0.959,0.955
XGBoost,0.938,0.964,1.000,0.987,0.982,0.980
XGBoost Tweedie,0.947,0.959,0.987,1.000,0.984,0.983
LightGBM,0.960,0.959,0.982,0.984,1.000,0.992
CatBoost,0.965,0.955,0.980,0.983,0.992,1.000


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\07_CorrelacionResiduales.csv

Correlación media entre residuales: 0.965
Correlación mínima: 0.916  (par: ('Ridge', 'Random Forest'))
Correlación máxima: 0.992  (par: ('LightGBM', 'CatBoost'))

>>> DIAGNÓSTICO: diversidad limitada. Ganancia esperada pequeña pero no nula.


La correlación entre residuales es, por sí sola, un resultado reportable. Si los modelos base resultan
redundantes, la conclusión no es que el ensamble "no funcionó", sino que **el límite del desempeño no
está en el algoritmo sino en la información disponible**: todos los modelos extraen esencialmente la
misma señal de las mismas variables, y lo que queda sin explicar depende de factores que el registro
administrativo no captura. Ese argumento sostiene directamente la respuesta a la pregunta sobre en qué
medida resulta modelable la DEH.

## 6. Blending: combinación lineal con pesos no negativos

Los pesos se estiman sobre una partición de validación separada del entrenamiento, nunca sobre el
conjunto de prueba: ajustarlos con las predicciones de prueba equivaldría a optimizar contra la métrica
que se pretende estimar. Se restringen a valores no negativos para que el resultado sea interpretable
como una ponderación de modelos.

In [7]:
# Partición interna de validación, tomada del ENTRENAMIENTO
X_fit, X_val, y_fit, y_val = train_test_split(X_tr, y_tr, test_size=0.20, random_state=SEMILLA)

pred_val, pred_test_bl = {}, {}
for nombre, modelo in modelos_base.items():
    m = clone(modelo)
    m.fit(X_fit, y_fit)
    pred_val[nombre] = np.clip(m.predict(X_val), 1, None)
    pred_test_bl[nombre] = np.clip(m.predict(X_test), 1, None)
    print(f"Reajustado para blending: {nombre}")

P_val = pd.DataFrame(pred_val)
P_test = pd.DataFrame(pred_test_bl)

# Pesos no negativos, sin intercepto
combinador = LinearRegression(positive=True, fit_intercept=False)
combinador.fit(P_val, y_val)

pesos = pd.Series(combinador.coef_, index=P_val.columns).sort_values(ascending=False)
print("\nPesos estimados:")
display(pesos.round(4).to_frame("peso"))
guardar_tabla(pesos.to_frame("peso"), "PesosBlending")

pred_blend = np.clip(combinador.predict(P_test), 1, None)
pred_promedio = np.clip(P_test.mean(axis=1).to_numpy(), 1, None)

Reajustado para blending: Ridge
Reajustado para blending: Random Forest
Reajustado para blending: XGBoost
Reajustado para blending: XGBoost Tweedie
Reajustado para blending: LightGBM
Reajustado para blending: CatBoost

Pesos estimados:


,peso
XGBoost Tweedie,0.4308
LightGBM,0.3211
CatBoost,0.1320
Random Forest,0.1009
Ridge,0.0352
XGBoost,0.0077


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\07_PesosBlending.csv


Los pesos son informativos por sí mismos: un peso nulo significa que ese modelo no aporta nada que los
demás no aporten ya, y un peso muy concentrado en un solo modelo indica que la combinación se reduce,
en la práctica, a ese modelo.

## 7. Stacking con predicciones fuera de pliegue

El meta-modelo debe entrenarse con predicciones que los modelos base no hayan visto durante su propio
ajuste. De lo contrario aprendería sobre predicciones sobreajustadas y asignaría una confianza excesiva
a los modelos con mayor capacidad de memorización.

Se generan predicciones fuera de pliegue mediante validación cruzada de 5 particiones sobre el conjunto
de entrenamiento. Este es el paso costoso del cuaderno.

In [8]:
N_PLIEGUES = 5
kf = KFold(n_splits=N_PLIEGUES, shuffle=True, random_state=SEMILLA)

oof = {}
for nombre, modelo in modelos_base.items():
    t0 = time.time()
    oof[nombre] = cross_val_predict(modelo, X_tr, y_tr, cv=kf, n_jobs=1)
    print(f"Fuera de pliegue completado: {nombre}  ({time.time()-t0:.0f}s)")

OOF = pd.DataFrame(oof, index=X_tr.index).clip(lower=1)
guardar_tabla(OOF.describe().T, "ResumenPrediccionesOOF")

Fuera de pliegue completado: Ridge  (6s)
Fuera de pliegue completado: Random Forest  (1925s)
Fuera de pliegue completado: XGBoost  (78s)
Fuera de pliegue completado: XGBoost Tweedie  (81s)
Fuera de pliegue completado: LightGBM  (20s)
Fuera de pliegue completado: CatBoost  (63s)
Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\07_ResumenPrediccionesOOF.csv


In [9]:
# Los modelos base se reentrenan sobre el total del entrenamiento para predecir sobre prueba
P_test_stack = pd.DataFrame({n: pred_test[n] for n in modelos_base})

meta_modelos = {
    "Meta: Ridge": Ridge(alpha=1.0),
    "Meta: Lineal no negativa": LinearRegression(positive=True, fit_intercept=False),
    "Meta: XGBoost superficial": XGBRegressor(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        tree_method="hist", random_state=SEMILLA, n_jobs=-1),
}

pred_stack = {}
for nombre, meta in meta_modelos.items():
    meta.fit(OOF, y_tr)
    pred_stack[nombre] = np.clip(meta.predict(P_test_stack), 1, None)
    joblib.dump(meta, MODELOS_PATH / f"07_{nombre.replace(': ', '_').replace(' ', '_')}.joblib")
    print(f"Meta-modelo entrenado: {nombre}")

Meta-modelo entrenado: Meta: Ridge
Meta-modelo entrenado: Meta: Lineal no negativa
Meta-modelo entrenado: Meta: XGBoost superficial


## 8. Comparación final

In [10]:
comparacion = []

for nombre, p in pred_test.items():
    comparacion.append({"estrategia": f"Individual — {nombre}", **metricas(y_test, p)})

comparacion.append({"estrategia": "Blending — promedio simple", **metricas(y_test, pred_promedio)})
comparacion.append({"estrategia": "Blending — pesos no negativos", **metricas(y_test, pred_blend)})
for nombre, p in pred_stack.items():
    comparacion.append({"estrategia": f"Stacking — {nombre}", **metricas(y_test, p)})

comp_df = pd.DataFrame(comparacion).sort_values("R2", ascending=False).reset_index(drop=True)
comp_df["ganancia_R2_vs_mejor_base"] = comp_df["R2"] - R2_MEJOR_BASE
display(comp_df.round(4))
guardar_tabla(comp_df, "ComparacionEnsambles")

ganador = comp_df.iloc[0]
print()
print(f"Mejor estrategia: {ganador['estrategia']}  (R2={ganador['R2']:.4f})")
print(f"Ganancia sobre el mejor modelo individual: {ganador['ganancia_R2_vs_mejor_base']:+.4f} de R2")

if ganador["ganancia_R2_vs_mejor_base"] < 0.01:
    print()
    print(">>> La ganancia es inferior a 0,01 de R2.")
    print(">>> Recomendación: conservar el modelo individual como objeto del análisis SHAP")
    print(">>> y reportar el ensamble solo como referencia de desempeño alcanzable.")

,estrategia,MAE,RMSE,R2,ganancia_R2_vs_mejor_base
0,Stacking — Meta: Ridge,2.9640,6.1557,0.4502,0.0047
1,Stacking — Meta: Lineal no negativa,2.9776,6.1568,0.4500,0.0045
2,Stacking — Meta: XGBoost superficial,2.9591,6.1765,0.4465,0.0009
3,Blending — pesos no negativos,2.9982,6.1792,0.4460,0.0005
4,Individual — XGBoost Tweedie,2.9208,6.1817,0.4456,0.0000
5,Blending — promedio simple,2.9923,6.2010,0.4421,-0.0035
6,Individual — LightGBM,3.0139,6.2021,0.4419,-0.0037
7,Individual — XGBoost,2.9869,6.2036,0.4416,-0.0039
8,Individual — CatBoost,3.0328,6.2421,0.4347,-0.0109
9,Individual — Random Forest,3.0266,6.3367,0.4174,-0.0281


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\07_ComparacionEnsambles.csv

Mejor estrategia: Stacking — Meta: Ridge  (R2=0.4502)
Ganancia sobre el mejor modelo individual: +0.0047 de R2

>>> La ganancia es inferior a 0,01 de R2.
>>> Recomendación: conservar el modelo individual como objeto del análisis SHAP
>>> y reportar el ensamble solo como referencia de desempeño alcanzable.


## 9. Desempeño por tramo de estancia

Incluso cuando la métrica global apenas se mueve, un ensamble puede redistribuir el error entre tramos.
Esta tabla verifica si la combinación mejora en la cola larga, que es el segmento donde los modelos
individuales fallan y el de mayor relevancia operativa.

In [11]:
TRAMOS = [(1, 2, "1-2 días"), (3, 6, "3-6 días"), (7, 13, "7-13 días"),
          (14, 29, "14-29 días"), (30, 119, "30-119 días"), (120, 120, "120+ (censurado)")]

candidatos = {
    f"Individual — {MEJOR_BASE}": pred_test[MEJOR_BASE],
    "Blending — pesos no negativos": pred_blend,
    "Stacking — Meta: Ridge": pred_stack["Meta: Ridge"],
}

filas = []
for nombre, p in candidatos.items():
    for lo, hi, etiqueta in TRAMOS:
        m = ((y_test >= lo) & (y_test <= hi)).to_numpy()
        if m.sum() == 0:
            continue
        filas.append({"estrategia": nombre, "tramo": etiqueta, "n": int(m.sum()),
                      "real_medio": float(y_test.to_numpy()[m].mean()),
                      "pred_media": float(p[m].mean()),
                      "MAE": mean_absolute_error(y_test.to_numpy()[m], p[m])})

tramos_df = pd.DataFrame(filas)
display(tramos_df.round(2))
guardar_tabla(tramos_df, "EnsamblesPorTramo")

,estrategia,tramo,n,real_medio,pred_media,MAE
0,Individual — XGBoost Tweedie,1-2 días,165259,1.56,3.11,1.60
1,Individual — XGBoost Tweedie,3-6 días,152033,4.06,4.92,1.84
2,Individual — XGBoost Tweedie,7-13 días,64491,9.16,8.31,3.42
3,Individual — XGBoost Tweedie,14-29 días,28338,18.94,12.49,8.07
4,Individual — XGBoost Tweedie,30-119 días,7658,46.35,21.46,26.55
5,Individual — XGBoost Tweedie,120+ (censurado),289,120.00,36.66,83.34
6,Blending — pesos no negativos,1-2 días,165259,1.56,3.29,1.77
7,Blending — pesos no negativos,3-6 días,152033,4.06,5.14,1.95
8,Blending — pesos no negativos,7-13 días,64491,9.16,8.59,3.39
9,Blending — pesos no negativos,14-29 días,28338,18.94,12.80,7.88


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\07_EnsamblesPorTramo.csv


## 10. Conclusiones del cuaderno

Elementos a documentar en el Capítulo 4:

1. **La correlación entre residuales.** Es el resultado conceptualmente más relevante del cuaderno.
   Una correlación alta sostiene el argumento de que el límite del desempeño proviene de la información
   disponible y no de la elección del algoritmo.
2. **Los pesos del blending.** Los modelos con peso nulo son redundantes respecto del conjunto; los
   pesos concentrados indican que la combinación equivale prácticamente a un modelo único.
3. **La magnitud de la ganancia.** Si es inferior a 0,01 de $R^2$, la conclusión defendible es que el
   ensamble no justifica la pérdida de interpretabilidad que supone.
4. **La tabla por tramos.** Verificar si la ganancia, de existir, se concentra en la cola larga o
   simplemente refina los tramos que ya se predecían bien.
5. **El desempeño de CatBoost**, si estuvo disponible, comparado con el XGBoost sobre variables
   codificadas mediante *Target Encoding*. Una diferencia apreciable indicaría que el tratamiento de
   las variables de alta cardinalidad es un factor relevante, lo que conectaría con la discusión sobre
   las limitaciones del esquema de codificación adoptado en la capa Gold.

**Tablas exportadas:** `07_DesempenoModelosBase`, `07_CorrelacionResiduales`, `07_PesosBlending`,
`07_ResumenPrediccionesOOF`, `07_ComparacionEnsambles`, `07_EnsamblesPorTramo`.